# TPC-H Collector Demo


In [1]:
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option('display.max_colwidth', None)
pd.options.display.max_columns = None
pd.options.display.max_rows = None
pd.options.display.float_format = '{:.2f}'.format

from bexhoma import collectors

%matplotlib inline

## Functions for Nice Plots

In [2]:
from notebookutils import *

# Collect Results

In [3]:
#path = r"D:\data\benchmarks"
path = r"/data/benchmarks"
filename_prefix = "demo_"
b_plot_save = False
b_skip_plots = True

In [4]:
codes = [
    "1776765494",
    "1776766994"
]

codes

['1776765494', '1776766994']

In [5]:
collect = collectors.dbmsbenchmarker(path, codes)

Path does not exist: /data/benchmarks/1776765494
NoneType: None


FileNotFoundError: [Errno 2] No such file or directory: '/connections.config'

# Get all Metrics Metadata

## Metrics Names and Types

Metrics that are derived from monitoring

In [ ]:
collect.get_metrics_metadata()

## Names of Monitored Components

Names of components and their phases

In [ ]:
collect.get_monitored_components()

# Get Connection Infos

List of states of SUTs, containing loading infos.

In [ ]:
df=collect.get_connections()

In [ ]:
df.T

## Test for Header Informations

These infos are used to relate connection infos to benchmarking results.

In [ ]:
df[['phase', 'code', 'connection', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']].T

## Get Non-Constant Attributes

These attributes are (most probably) used to analyze performance.

In [ ]:
collectors.get_non_constant(df).T

# Monitoring Aggregated per Phase

In [ ]:
df = collect.get_monitoring_aggregated_per_phase("stream")
df.T

## With Metadata

Connection infos are added.

In [ ]:
df = collect.add_metadata(df)
df.T

## Test for Header Informations

These infos are used to relate connection infos to benchmarking results.

In [ ]:
df[['phase', 'code', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']].T

# Time Series of a Single Metric for a Single Experiment

This is particularly important if you require a customized aggregation or wish to monitor a component over time.

In [ ]:
metric = 'total_cpu_memory'
code = codes[0]
df = collect.get_monitoring_timeseries_per_phase(code, metric=metric, component='stream')
df.T

## With Metadata

Connection infos are added.

In [ ]:
df = collect.add_metadata(df)
df.T

## Test for Header Informations

These infos are used to relate connection infos to benchmarking results.

In [ ]:
df[['phase', 'code', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']].head().T

# Time Series of a Single Metric for All Experiments

This is particularly important if you require a customized aggregation or wish to monitor a component over time.
Data is stacked here (long format).

In [ ]:
metric = 'total_cpu_memory'
code = codes[0]
df = collect.get_monitoring_timeseries_all(metric=metric)
df.head()

## Test for Header Informations

These infos are used to relate connection infos to benchmarking results.

In [ ]:
df[['timestamp', 'phase', 'value']].head(8)

## With Metadata

Connection infos are added.

In [ ]:
df=collect.add_metadata(df)
df = df.sort_values(['timestamp', 'phase'])
df[['timestamp', 'phase', 'value']].head(8)

## Test for Header Informations

These infos are used to relate connection infos to benchmarking results.

In [ ]:
df[['phase', 'code', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']].head().T

## Custom Aggregation and Scale

We here aggregate CPU utilization by max per phase.

In [ ]:
metric = 'total_cpu_util'

df_performance_series = collect.get_monitoring_timeseries_all(metric)

df_agg = (
    df_performance_series.groupby(["code", "experiment_run", "client", "type_tenants", "num_tenants"])["value"]
      .max()
      .reset_index()
)
df_agg.index.name = 'Max ' + collect.df_metrics.loc[metric]['title']
df_agg

# Performance Results per Connection

In [ ]:
df_performance = collect.get_performance_per_connection()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.T

## With Metadata

Connection infos are added.

In [ ]:
df = collect.add_metadata(df_performance)
df.T

# Performance Results per Phase

In [ ]:
df_performance = collect.get_performance_aggregated_per_phase()

df_performance_first = df_performance[df_performance['client']==1]
df_performance_second = df_performance[df_performance['client']==2]

df_performance.dropna(inplace=True)

In [ ]:
df_performance.T

## With Metadata

Connection infos are added.

In [ ]:
df = collect.add_metadata(df_performance)
df.T

## Test for Header Informations

These infos are used to relate connection infos to benchmarking results.

In [ ]:
df[['phase', 'code', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']].T

## Get Non-Constant Attributes

These attributes are (most probably) used to analyze performance.

In [ ]:
collectors.get_non_constant(df).T

# Performance at Ingestion per Run

This shows loading performance per experiment run.

In [ ]:
df = collect.get_loading_per_run()
df.T

## With Metadata

Connection infos are added.

In [ ]:
df = collect.add_metadata(df)
df.T

# Combine Performance and Monitoring

This includes filtering for the first client and incorporating efficiency metrics to track performance relative to resource usage.

In [ ]:
client = 1

df_performance_monitoring = collect.get_monitoring_aggregated_per_phase(type="stream")
df_performance = collect.get_performance_aggregated_per_phase()
merged_df = pd.merge(df_performance, df_performance_monitoring, left_index=True, right_index=True, how='left')

merged_df = merged_df[merged_df['client'] == client]

#merged_df['E_Tpx'] = merged_df['Goodput (requests/second)'] / merged_df['CPU Utilization Time [s]'] * 600.
#merged_df['E_Lat'] = 1./np.sqrt(merged_df['Latency Distribution.Average Latency (microseconds)']*merged_df['CPU Utilization Time [s]']/1E6)
#merged_df['E_RAM'] = (merged_df['Goodput (requests/second)']) / merged_df['Memory Usage [MiB]']
merged_df.T

## With Metadata

Connection infos are added.

In [ ]:
df = collect.add_metadata(merged_df)
df.T

## Test for Header Informations

These infos are used to relate connection infos to benchmarking results.

In [ ]:
df[['phase', 'code', 'configuration', 'experiment_run', 'client', 'type_tenants', 'num_tenants', 'vol_tenants']].T

# Show Warnings per Connection

In [ ]:
df = collect.get_total_warnings(query_titles=False)
df.T

## Show Only Existing Warnings

For demonstration we insert a warning.

In [ ]:
df.loc['1776765494-PostgreSQL-BHT-8-1-1-1','Q1'] = True
df.T

In [ ]:
num_warnings = df.sum().sum()
if num_warnings > 0:
    # remove only False rows
    df = df[~(df == False).all(axis=1)]
    print(df.T)
else:
    print("No warnings")

# Show Warnings per Connection with Query Titles

In [ ]:
df = collect.get_total_warnings(query_titles=True)
df.T

In [ ]:
num_warnings = df.sum().sum()
if num_warnings > 0:
    # remove only False rows
    df = df[~(df == False).all(axis=1)]
    print(df.T)
else:
    print("No warnings")

# Show Errors per Connection with Query Titles

In [ ]:
df = collect.get_total_errors(query_titles=True)
df.T

## Show Only Existing Errors

In [ ]:
num_errors = df.sum().sum()
if num_errors > 0:
    # remove only False rows
    df = df[~(df == False).all(axis=1)]
    print(df.T)
else:
    print("No errors")

# Show Latencies per Query and Connection with Query Titles

In [ ]:
df = collect.get_query_latencies(query_titles=True)
df.T

In [ ]:
zip_all_results(path, codes)